In [2]:
import json

# 1. Open and read the config file
# Since 'prototype.ipynb' is in 'ONXX/notebooks/', we go up one folder '../' to find 'config.json'
with open("../config.json", "r") as file:
    config_data = json.load(file)

# 2. Extract our settings
num_actions = config_data["num_actions"]
num_observations = config_data["num_observations"]
gamma = config_data["gamma"]

# 3. Print them to prove it worked!
print(f"Configuration Loaded Successfully!")
print(f"Number of Actions: {num_actions}")
print(f"Discount Factor (Gamma): {gamma}")


Configuration Loaded Successfully!
Number of Actions: 4
Discount Factor (Gamma): 0.95


In [3]:
import random

class CustomDiscreteActionSpace:
    """
    A simple manager for our actions.
    It configures itself using the limit we loaded from our JSON file.
    """
    def __init__(self, limit:int):
        self.n = limit # action space

    def sample(self) -> int:
        """Selects a random action. Perfect for training exploration."""
        return random.randint(0, self.n - 1)

    def contains(self, action) -> bool:
        """Checks if a given action is actually valid."""
        if not isinstance(action, int):
            return False
        return 0 <= action < self.n

In [3]:
class CustomContinuousObservationSpace:
    """
    A simple manager for our agent's observations (what it sees).
    Configures itself dynamically using the limit from our JSON file.
    """
    def __init__(self, num_variables:int):
        self.shape = num_variables

    def contains(self, observations) -> bool:
        # is it actually a lis?
        if not isinstance(observations, list):
            return False

        # is the length equal
        if len(observations) != num_observations:
            return False

        # if it contains the data type selected (typically a float)
        for value in observations:
            if not isinstance(value, float):
                return False

        return True

In [4]:
import json
import zmq

class CustomRLEnv:
    """
    The master environment that ties our spaces together.
    """
    def __init__(self, config_path: str):
        # 1. Load JSON settings file
        with open(config_path, "r") as file:
            self.config = json.load(file)
            
        # 2. Extract configuration limits
        num_actions = self.config["num_actions"]
        num_obs = self.config["num_observations"]
        
        # 3. Create ovalidation spaces using the above classes 
        self.action_space = CustomDiscreteActionSpace(num_actions)
        self.observation_space = CustomContinuousObservationSpace(num_obs)

        # 4. Create ZeroMQ context and req socket
        self.context = zmq.Context()
        self.socket = self.context.socket(zmq.REQ)
        self.socket.connect("tcp://localhost:5555")
        
        # 5. Episode state variables
        self.done = False

    def reset(self) -> list:
        """Resets the world to the starting line."""
        self.done = False
        
        # For now, we return a mock starting observation of 6 zeros
        # [rel_x, rel_y, rel_z, vel_x, vel_y, vel_z]
        initial_state = [0.0] * self.observation_space.shape
        return initial_state

    def step(self, action:int) -> tuple:
        if not self.action_space.contains(action):
            raise ValueError("InvalidAction!")

        self.socket.send_string(str(action))

        reply = self.socket.recv_string()

        reward = 0.7

        self.done = False

        return reply, reward, self.done

In [5]:
import os
import sys
import json
import zmq

# Ensure Python can locate the RL module inside the notebooks folder
sys.path.append(os.path.abspath("."))

# Import your generated FlatBuffers schema
import RL.EnvState as EnvState

class CustomRLEnv:
    """
    The master environment that ties custom spaces, JSON configs,
    and high-speed FlatBuffers binary IPC together!
    """
    def __init__(self, config_path: str, address="tcp://localhost:5555"):
        # 1. Load JSON configuration parameters
        with open(config_path, "r") as file:
            self.config = json.load(file)
            
        self.num_actions = self.config["num_actions"]
        self.num_obs = self.config["num_observations"]
        self.gamma = self.config.get("gamma", 0.95)
        
        # 2. Instantiate your custom spaces (keeps your original gym mechanics)
        self.action_space = CustomDiscreteActionSpace(self.num_actions)
        self.observation_space = CustomContinuousObservationSpace(self.num_obs)

        # 3. Create ZeroMQ context and REQ socket
        self.context = zmq.Context()
        self.socket = self.context.socket(zmq.REQ)
        self.socket.connect(address)
        print(f"Custom Gym online! Connected to C++ environment at {address}")
        
        self.done = False

    def reset(self) -> dict:
        """
        Resets the C++ simulation to the starting line and returns
        the initial environment observation unpacked via FlatBuffers.
        """
        self.done = False
        
        # Send a special reset control signal to C++
        self.socket.send_string("RESET")
        
        # Block and receive the fresh initial state buffer
        raw_bytes = self.socket.recv()
        
        # Unpack and return the initial observation
        observation = self._unpack_observation(raw_bytes)
        return observation

    def step(self, action: int) -> tuple:
        """
        Validates the action, sends the index to C++, and decodes
        the physical state, reward, and terminal flags via FlatBuffers.
        """
        # 1. Validate action using your custom space
        if not self.action_space.contains(action):
            raise ValueError(f"Invalid Action! Action {action} is out of bounds.")

        # 2. Send the validated action index to C++
        self.socket.send_string(str(action))

        # 3. Block and receive the raw binary payload
        raw_bytes = self.socket.recv()

        # 4. Zero-Parsing Map
        state = EnvState.EnvState.GetRootAsEnvState(raw_bytes, 0)
        
        # 5. Extract structural data & reward/done parameters directly from memory
        observation = self._unpack_observation(raw_bytes)
        reward = state.Reward()
        self.done = state.Done()

        return observation, reward, self.done

    def _unpack_observation(self, raw_bytes: bytes) -> dict:
        """Helper to cleanly map FlatBuffers memory fields to observation layout."""
        state = EnvState.EnvState.GetRootAsEnvState(raw_bytes, 0)
        pos = state.Position()
        vel = state.Velocity()
        
        # Maps coordinates to 6 observation variables
        # [rel_x, rel_y, rel_z, vel_x, vel_y, vel_z]
        return {
            "position": (pos.X(), pos.Y(), pos.Z()),
            "velocity": (vel.X(), vel.Y(), vel.Z())
        }

    def close(self):
        """Cleans up the networking sockets."""
        self.socket.close()
        self.context.term()
        print("Sockets closed.")

In [6]:
# 1. Instantiate env using your original JSON file!
env = CustomRLEnv(config_path="../config.json")

# 2. Trigger our new FlatBuffers-based RESET handshake
print("\n--- Resetting Environment ---")
initial_obs = env.reset()
print(f"Starting Observation Vector: {initial_obs}\n")

# 3. Simulate taking 3 actions using integer indices (e.g. 0 or 1)
test_actions = [0]

print("--- Stepping Environment ---")
for act in test_actions:
    print(f"Agent Action: {act}")
    obs, reward, done = env.step(act)
    print(f"Observation: {obs}")
    print(f"Reward     : {reward}")
    print(f"Done       : {done}\n")

env.close()

NameError: name 'CustomContinuousObservationSpace' is not defined